# Model

## Imports & Load Datasets

In [158]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, FunctionTransformer, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import mean_absolute_error, median_absolute_error, r2_score
from sklearn.model_selection import KFold, cross_val_score

df_train = pd.read_csv('../data/work/train.csv')
df_test  = pd.read_csv('../data/work/test.csv')
X_train = df_train.drop(columns=['total'])
y_train = df_train['total'].values
X_test  = df_test.drop(columns=['total'])
y_test  = df_test['total'].values

## Normalize and Pipelines

In [ ]:
cat_cols = ["address", "district", "type"]
num_cols = ["bedrooms", "garage"]
area_col = ["area"]  

for df in (X_train, X_test):
    for c in cat_cols:
        df[c] = df[c].astype(str).str.strip().str.lower()


p99_area = np.nanpercentile(X_train["area"].to_numpy(), 99)

Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

area_tree = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("clip99", FunctionTransformer(
        lambda X: np.clip(X, None, p99_area),
        feature_names_out="one-to-one"
    )),
])

num_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

area_lin = Pipeline([
    ("imp", SimpleImputer(strategy="mean")),  
    ("clip99", FunctionTransformer(
        lambda X: np.clip(X, None, p99_area),
        feature_names_out="one-to-one"
    )),
    ("scaler", StandardScaler()),
])

cat_pipe = Pipeline([
    ("imp", SimpleImputer(strategy="constant", fill_value="missing")),
    ("ohe", OneHotEncoder(handle_unknown="ignore", min_frequency=0.01)),
])

ct_tree = ColumnTransformer(
    transformers=[
        ("num", num_pipe, num_cols),
        ("area",      area_tree, area_col),  
        ("cat",       cat_pipe,  cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

ct_linear = ColumnTransformer(
    transformers=[
        ("num",  num_pipe,  num_cols),
        ("area", area_lin, area_col),        
        ("cat",  cat_pipe, cat_cols),
    ],
    remainder="drop",
    sparse_threshold=0.3,
)

Xt_tree_tr = ct_tree.fit_transform(X_train)
Xt_tree_te = ct_tree.transform(X_test)
Xt_lin_tr  = ct_linear.fit_transform(X_train)
Xt_lin_te  = ct_linear.transform(X_test)

print("Tree CT -> train/test shapes:", Xt_tree_tr.shape, Xt_tree_te.shape)
print("Linear CT -> train/test shapes:", Xt_lin_tr.shape, Xt_lin_te.shape)


Tree CT -> train/test shapes: (9325, 23) (2332, 23)
Linear CT -> train/test shapes: (9325, 23) (2332, 23)


In [160]:
def eval_predictions(y_true, y_pred):
    return {
        "MAE":   mean_absolute_error(y_true, y_pred),
        "MedAE": median_absolute_error(y_true, y_pred),
        "R2":    r2_score(y_true, y_pred),
    }

def eval_baseline_median(y_true, train_median):
    y_pred = np.full_like(y_true, fill_value=float(train_median), dtype=float)
    return eval_predictions(y_true, y_pred)

## Build the Linear Model

In [161]:
ridge = Ridge(random_state=42)

pipe_ridge = Pipeline(steps=[
    ("preprocess", ct_linear),
    ("model", ridge)
])

## Cross Validation

In [162]:
cv = KFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(
    pipe_ridge,
    X_train, y_train,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

cv_mae = -cv_scores
print(f"CV MAE (5-fold): mean={cv_mae.mean():.2f}  std={cv_mae.std():.2f}")

CV MAE (5-fold): mean=1413.04  std=31.59


## Getting the MAE, MedAE and R² Metrics

In [163]:
pipe_ridge.fit(X_train, y_train)
y_pred_test = pipe_ridge.predict(X_test)

res_ridge_test = eval_predictions(y_test, y_pred_test)
print(res_ridge_test)

{'MAE': 1439.4297104123898, 'MedAE': 1035.9551471277728, 'R2': 0.6135404677420908}


## Top 15 coefficients

In [164]:
ridge_fitted = pipe_ridge.named_steps["model"]

feature_names = pipe_ridge.named_steps["preprocess"].get_feature_names_out()

coefs = pd.DataFrame({
    "feature": feature_names,
    "coef": ridge_fitted.coef_,
})
coefs["abs_coef"] = coefs["coef"].abs()

top15 = coefs.sort_values("abs_coef", ascending=False).head(15)
top15

,feature,coef,abs_coef
7,cat__district_cerqueira césar,3150.212104,3150.212104
9,cat__district_jardim paulista,2573.111671,2573.111671
2,area__area,1979.123987,1979.123987
20,cat__type_casa,-1529.578946,1529.578946
12,cat__district_pinheiros,1429.830556,1429.830556
5,cat__district_brás,-1246.013965,1246.013965
6,cat__district_centro,-1244.380871,1244.380871
10,cat__district_liberdade,-1212.913970,1212.913970
11,cat__district_mooca,-1185.960444,1185.960444
16,cat__district_vila andrade,-983.925118,983.925118


## Build the Random Forest Model

In [165]:
random_forest = RandomForestRegressor(random_state=42) 

pipe_forest = Pipeline(steps=[
    ("preprocess", ct_tree),
    ("model", random_forest)
])

## Cross Validation

In [166]:
cv_scores_forest = cross_val_score(
    pipe_forest,
    X_train, y_train,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

cv_mae_forest = -cv_scores_forest
print(f"CV MAE (5-fold): mean={cv_mae_forest.mean():.2f} std={cv_mae_forest.std():.2f}")

CV MAE (5-fold): mean=1244.22 std=46.29


## Getting the MAE, MedAE and R² Metrics

In [167]:
pipe_forest.fit(X_train, y_train)
y_pred_test_forest = pipe_forest.predict(X_test)

res_forest_test = eval_predictions(y_test, y_pred_test_forest)
print(res_forest_test)

{'MAE': 1285.9692668854295, 'MedAE': 762.3311787714767, 'R2': 0.6479025803094671}


## Top 15 coefficients

In [168]:
forest_fitted  = pipe_forest.named_steps["model"]
preprocess     = pipe_forest.named_steps["preprocess"]

feature_names  = preprocess.get_feature_names_out()

imp = forest_fitted.feature_importances_

coefs = pd.DataFrame({"feature": feature_names, "importance": imp})
top15 = coefs.sort_values("importance", ascending=False).head(15)
top15

,feature,importance
2,area__area,0.651369
1,num__garage,0.101228
20,cat__type_casa,0.091463
0,num__bedrooms,0.046947
19,cat__type_apartamento,0.029699
7,cat__district_cerqueira césar,0.013322
9,cat__district_jardim paulista,0.013313
18,cat__district_infrequent_sklearn,0.012449
12,cat__district_pinheiros,0.007133
8,cat__district_consolação,0.005789


## Linear Tunnig

In [169]:
y_train = pd.to_numeric(y_train, errors="coerce")

parameters = {"model__alpha": [0.01, 0.1, 0.3, 1, 3, 10]}

grid_search = GridSearchCV(
    estimator = pipe_ridge,
    param_grid = parameters,
    scoring="neg_mean_absolute_error",
    cv=5,
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

best_mae = -grid_search.best_score_  
best_params = grid_search.best_params_

print(f"Best MAE: {best_mae:.2f}")
print("Best Parameters:", best_params)

Best MAE: 1411.03
Best Parameters: {'model__alpha': 10}


## Cross Validation

In [170]:
ridge_tunned = Ridge(alpha=10, random_state=42)

pipe_ridge_tunned = Pipeline(steps=[
    ("preprocess", ct_linear),
    ("model", ridge_tunned)
])

cv_scores_tunned = cross_val_score(
    pipe_ridge_tunned,
    X_train, y_train,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

cv_mae_tunned = -cv_scores_tunned
print(f"CV MAE (5-fold): mean={cv_mae_tunned.mean():.2f}  std={cv_mae_tunned.std():.2f}")

CV MAE (5-fold): mean=1411.87  std=31.48


## Getting the MAE, MedAE and R² Metrics

In [171]:
pipe_ridge_tunned.fit(X_train, y_train)
y_pred_test_tunned = pipe_ridge_tunned.predict(X_test)

res_ridge_test = eval_predictions(y_test, y_pred_test_tunned)
print(res_ridge_test)

{'MAE': 1438.5646147400716, 'MedAE': 1035.6992175410958, 'R2': 0.6138832980858586}


## Forest Tunning

In [172]:
y_train = pd.to_numeric(y_train, errors="coerce")

random_forest = RandomForestRegressor(random_state=42, n_jobs=-1)

pipe_forest_tunned = Pipeline(steps=[
    ("preprocess", ct_tree),       
    ("model", random_forest)
])


param_dist = {
    "model__n_estimators": [300, 600, 900],
    "model__max_depth": [None, 10, 20, 30],
    "model__min_samples_leaf": [1, 2, 4],
    "model__min_samples_split": [2, 5, 10],
    "model__max_features": ["sqrt", 0.5],
    "model__bootstrap": [True],
}

random_search_forest = RandomizedSearchCV(
    estimator=pipe_forest_tunned,                   
    param_distributions=param_dist,                
    n_iter=20,
    scoring="neg_mean_absolute_error",             
    cv=5,
    random_state=42,
    n_jobs=-1,
    error_score=np.nan                              
)

random_search_forest.fit(X_train, y_train)

best_mae_forest   = -random_search_forest.best_score_
best_params_forest = random_search_forest.best_params_
best_forest_pipe   = random_search_forest.best_estimator_

y_pred = best_forest_pipe.predict(X_test)


In [173]:
print(f"Best MAE (CV): {best_mae_forest:.2f}")
print("Best Parameters:", best_params_forest)

Best MAE (CV): 1186.96
Best Parameters: {'model__n_estimators': 300, 'model__min_samples_split': 10, 'model__min_samples_leaf': 4, 'model__max_features': 0.5, 'model__max_depth': 20, 'model__bootstrap': True}


## Cross Validation

In [174]:
cv_scores_tunned = cross_val_score(
    pipe_forest_tunned,
    X_train, y_train,
    scoring="neg_mean_absolute_error",
    cv=cv,
    n_jobs=-1
)

cv_mae_tunned = -cv_scores_tunned
print(f"CV MAE (5-fold): mean={cv_mae_tunned.mean():.2f}  std={cv_mae_tunned.std():.2f}")

CV MAE (5-fold): mean=1244.22  std=46.29


## Getting the MAE, MedAE and R² Metrics

In [175]:
y_pred_test_forest_tuned = best_forest_pipe.predict(X_test)
res_forest_test_tuned = eval_predictions(y_test, y_pred_test_forest_tuned)
print(res_forest_test_tuned)

{'MAE': 1233.6758867962042, 'MedAE': 752.9690466213201, 'R2': 0.685343388425256}
